The main script of the pipeline is in `pipeline/run_script.py`.
It runs in the following stages.

In [ ]:
from pathlib import Path
from pipeline.utils import setup_pipeline

config = setup_pipeline('config.yaml')

17:34:57 | INFO     |          |          | ================================================================================
17:34:57 | INFO     | setup    |          | Run:             test
17:34:57 | INFO     | setup    |          | Date:            2026-03-26 17:34:57
17:34:57 | INFO     | setup    |          | Log directory:   /Users/iii9781/pipeline/tmp/test/logs
17:34:57 | INFO     |          |          | ================================================================================
17:34:57 | WARNING  | env      |          | GPU: Not available


### Load probes
Load all probes and the ADC from OpenEphys session folders. Each Probe object contain recordings (`.dat` file), TTL sync events, and timestamps. `probe_filter` argument can be used to skip probes.

In [ ]:
from pipeline.operations import load_probes

probes = load_probes(["/path/to/session1", "/path/to/session2"], probe_filter=None)

17:34:59 | INFO     | load     |          | Found ADC (ID: 0)
17:34:59 | INFO     | load     | ADC      | Loading 2 session(s)
17:34:59 | INFO     | load     | ADC      | --------------------------------------------------------------------------------
17:34:59 | INFO     | load     | ADC      | Mouse10_20260210_795to2220_Staggered_AllenScenes
17:34:59 | INFO     | load     | ADC      |   binary: 1 recording(s) · 12 ch · 30.30 kHz · 19973542 samples (457.16 MB) · (10m 59.2s)
17:34:59 | INFO     | load     | ADC      | --------------------------------------------------------------------------------
17:34:59 | INFO     | load     | ADC      | Mouse10_20260210_795to2220_Staggered_FFGrating
17:34:59 | INFO     | load     | ADC      |   binary: 1 recording(s) · 12 ch · 30.30 kHz · 30533128 samples (698.85 MB) · (16m 47.7s)
17:34:59 | SUCCESS  | load     | ADC      | Loaded 2 recording(s)
17:34:59 | SUCCESS  |          | ADC      | load_sessions completed in 86.78ms
17:34:59 | INFO     | load

### Concatenate recordings

Multi-session recordings are concatenated into a single binary file per probe using SpikeInterface.

In [ ]:
from pipeline.operations import concatenate_probes

concatenate_probes(probes, config)

### Sorting

Run Kilosort4 on each probe's concatenated recording.

In [ ]:
from pipeline.operations import sort_probes

probe_paths = {
    name: Path(config['local_output']) / name / 'concat'
    for name in probes if name != 'OneBox-ADC'
}

sort_probes(probe_paths, config)

### Downsample

Decimate the concatenated recording to `target_fs`. Skip if `target_fs` isn't given.

In [ ]:
from pipeline.operations import downsample_probes

downsample_probes(probe_paths, config)

### Synchronize to ADC

Interpolate spike timestamps from probe time to ADC time.

In [ ]:
from pipeline.operations import synchronize_probes

synchronize_probes(probes, config)

### Copy to remote

The `copy_mode` option controls how existing files are handled when copying to remote storage:
- `newer` — Only overwrite if local file is newer (default)
- `prompt` — Prompt before overwriting each file
- `skip-all` — Skip all existing files
- `all` — Overwrite all existing files

In [ ]:
from pipeline.utils import copy_to_remote

copy_to_remote(
    local_path=config['local_output'],
    remote_path=config['remote_output'],
    overwrite_mode=config['copy_mode'],
    )
